In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Tạo chuỗi sin + noise
np.random.seed(42)

n_points = 600
t = np.arange(n_points)

series = np.sin(0.05 * t) + 0.1 * np.random.randn(n_points)

# Chuẩn hóa Min-Max
scaler = MinMaxScaler()
series_scaled = scaler.fit_transform(series.reshape(-1,1))

In [ ]:
seq_length = 20

X, y = [], []

for i in range(len(series_scaled) - seq_length):
    X.append(series_scaled[i:i+seq_length])
    y.append(series_scaled[i+seq_length])

X = np.array(X)
y = np.array(y)

# Chia dữ liệu theo thời gian
N = len(X)

train_end = int(0.70 * N)
val_end = int(0.85 * N)

X_train = torch.FloatTensor(X[:train_end])
y_train = torch.FloatTensor(y[:train_end])

X_val = torch.FloatTensor(X[train_end:val_end])
y_val = torch.FloatTensor(y[train_end:val_end])

X_test = torch.FloatTensor(X[val_end:])
y_test = torch.FloatTensor(y[val_end:])

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=False
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()

        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        return self.fc(out[:, -1, :])

In [ ]:
def train_model(model, epochs=100):

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    for epoch in range(epochs):

        model.train()

        for xb, yb in train_loader:

            optimizer.zero_grad()

            pred = model(xb)

            loss = criterion(pred, yb)

            loss.backward()

            optimizer.step()

        if (epoch + 1) % 10 == 0:

            model.eval()

            with torch.no_grad():
                val_pred = model(X_val)
                val_loss = criterion(val_pred, y_val)

            print(
                f"Epoch {epoch+1:3d} | "
                f"Val Loss = {val_loss:.6f}"
            )


def evaluate(model):

    model.eval()

    with torch.no_grad():
        pred = model(X_test).numpy()

    actual = y_test.numpy()

    mse = mean_squared_error(
        actual,
        pred
    )

    return mse, actual, pred

In [ ]:
print("===== TRAIN RNN =====")

rnn_model = RNN()

train_model(rnn_model)

rnn_mse, rnn_actual, rnn_pred = evaluate(
    rnn_model
)

print("RNN Test MSE =", rnn_mse)


print("\n===== TRAIN LSTM =====")

lstm_model = LSTMModel()

train_model(lstm_model)

lstm_mse, lstm_actual, lstm_pred = evaluate(
    lstm_model
)

print("LSTM Test MSE =", lstm_mse)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    lstm_actual,
    label="Actual"
)

plt.plot(
    lstm_pred,
    label="LSTM Prediction"
)

plt.legend()
plt.title("LSTM: Predicted vs Actual")
plt.show()

print("========== COMPARISON ==========")
print(f"RNN  Test MSE : {rnn_mse:.6f}")
print(f"LSTM Test MSE : {lstm_mse:.6f}")